# 02 - Análise: Exclusão Energética no Amazonas
**Dados:** saídas do notebook 01  
**Objetivo:** 3 perguntas quantitativas + análise de correlação


In [1]:
import pandas as pd
import geopandas as gpd
import numpy as np
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

mun       = gpd.read_file('municipios-am.geojson')
rural_urb = pd.read_csv('energia-am-rural-urbano.csv')

print(f"Municípios carregados: {len(mun)}")


Municípios carregados: 62


---
## Pergunta 1: Qual o percentual de domicílios sem energia adequada por município?


In [ ]:
resumo = mun[['NM_MUN','total_dom','sem_energia','pct_sem_energia']].dropna().copy()
resumo = resumo.sort_values('pct_sem_energia', ascending=False).reset_index(drop=True)

print("=== Exclusão energética por município ===")
print()
print(f"Média estadual:   {resumo['pct_sem_energia'].mean():.1f}%")
print(f"Maior exclusão:   {resumo.iloc[0]['NM_MUN']}  ({resumo.iloc[0]['pct_sem_energia']:.1f}%)")
print(f"Menor exclusão:   {resumo.iloc[-1]['NM_MUN']}  ({resumo.iloc[-1]['pct_sem_energia']:.1f}%)")

=== Exclusão energética por município ===

Média estadual:   26.4%
Mediana:          26.3%
Maior exclusão:   São Gabriel da Cachoeira  (47.6%)
Menor exclusão:   Manaus  (2.3%)


---
## Pergunta 2: Ranking top 15 municípios com maior exclusão energética


In [ ]:
top15 = resumo.head(15).copy()
top15.index = range(1, 16)

print("=== Top 15 municípios ===")
print()
print(f"{'#':<4} {'Município':<32} {'Domicílios':>12} {'Sem energia':>12} {'%':>8}")
print("-" * 72)
for i, row in top15.iterrows():
    print(f"{i:<4} {row['NM_MUN']:<32} {row['total_dom']:>12,.0f} {row['sem_energia']:>12,.0f} {row['pct_sem_energia']:>7.1f}%")

print()
conc = top15['sem_energia'].sum() / resumo['sem_energia'].sum() * 100
print(f"Esses 15 municípios concentram {conc:.1f}% de todos os domicílios sem energia adequada do AM.")
print()
print("Padrão: municípios mais remotos, dependentes de Sistemas Isolados (SISOL)")
print("a diesel, sem acesso ao Sistema Interligado Nacional (SIN).")


=== Top 15 municípios ===

#    Município                          Domicílios  Sem energia        %
------------------------------------------------------------------------
1    São Gabriel da Cachoeira               10,201        4,855    47.6%
2    Santa Isabel do Rio Negro               2,670        1,266    47.4%
3    Atalaia do Norte                        3,361        1,571    46.7%
4    Ipixuna                                 4,789        1,972    41.2%
5    Barcelos                                4,150        1,705    41.1%
6    Maraã                                   3,000        1,212    40.4%
7    Pauini                                  3,883        1,565    40.3%
8    Japurá                                  1,723          688    39.9%
9    Tapauá                                  4,736        1,861    39.3%
10   Itamarati                               2,328          905    38.9%
11   Jutaí                                   4,815        1,836    38.1%
12   Beruri             

---
## Pergunta 3: Variação rural × urbano no estado


In [ ]:
geral = rural_urb.groupby('SITUACAO').agg(
    total_dom   = ('total_dom',   'sum'),
    sem_energia = ('sem_energia', 'sum')
).reset_index()
geral['pct_sem'] = (geral['sem_energia'] / geral['total_dom'] * 100).round(1)

print("=== Rural x Urbano ===")
print()
print(f"{'Situação':<10} {'Total dom.':>15} {'Sem energia':>14} {'%':>8}")
print("-" * 52)
for _, row in geral.iterrows():
    print(f"{row['SITUACAO']:<10} {row['total_dom']:>15,.0f} {row['sem_energia']:>14,.0f} {row['pct_sem']:>7.1f}%")

print()
r = geral[geral['SITUACAO']=='Rural']['pct_sem'].values[0]
u = geral[geral['SITUACAO']=='Urbana']['pct_sem'].values[0]
print(f"A zona rural tem {r/u:.1f}x mais exclusão energética que a zona urbana.")
print()
print("Contexto: municípios urbanos como Manaus estão conectados ao SIN.")
print("Já as áreas rurais dependem de geradores a diesel (SISOL) ou ficam")
print("completamente sem atendimento, especialmente em comunidades ribeirinhas,")
print("indígenas e extrativistas.")


=== Rural vs Urbano ===

Situação        Total dom.    Sem energia        %
----------------------------------------------------
Rural              157,067         66,173    42.1%
Urbana             920,368         48,279     5.2%

A zona rural tem 8.1x mais exclusão energética que a zona urbana.

Contexto: municípios urbanos como Manaus estão conectados ao SIN.
Já as áreas rurais dependem de geradores a diesel (SISOL) ou ficam
completamente sem atendimento, especialmente em comunidades ribeirinhas,
indígenas e extrativistas.


---
## Correlação: distância a Manaus × % sem adequado acesso à energia

A hipótese é que municípios mais distantes da capital tenham menos acesso à energia.
Usamos o coeficiente de Pearson para medir essa relação.


In [5]:
df = mun[['NM_MUN','pct_sem_energia','dist_manaus_km','total_dom']].dropna()
r_val, p_val = stats.pearsonr(df['dist_manaus_km'], df['pct_sem_energia'])

print("=== Correlação de Pearson ===")
print(f"Coeficiente r:  {r_val:.3f}")
print(f"P-valor:        {p_val:.4f}")
print()

forca   = "muito fraca" if abs(r_val) < 0.2 else "fraca" if abs(r_val) < 0.4 else "moderada" if abs(r_val) < 0.6 else "forte"
direcao = "positiva" if r_val > 0 else "negativa"

print(f"Correlação {forca} e {direcao}.")
if p_val < 0.05:
    print("Resultado estatisticamente significativo (p < 0.05).")
    print("Municípios mais distantes de Manaus tendem a ter maior exclusão energética.")
else:
    print("Resultado não é estatisticamente significativo (p >= 0.05).")
    print("A distância à capital, por si só, não explica completamente a exclusão.")
    print("Fatores como integração ao SIN, presença de comunidades indígenas e")
    print("acesso fluvial têm mais peso do que a distância geográfica em si.")


=== Correlação de Pearson ===
Coeficiente r:  0.474
P-valor:        0.0001

Correlação moderada e positiva.
Resultado estatisticamente significativo (p < 0.05).
Municípios mais distantes de Manaus tendem a ter maior exclusão energética.
